# Tech Jobs Analyzer — Full Project Launcher

This notebook runs the complete project in one Colab session:

**Stage 1 → ChromaDB → Stage 2 → Gemini → Streamlit UI**

The dataset and ChromaDB are reused from Google Drive:
- Dataset: `/content/drive/MyDrive/postings.csv`
- ChromaDB: `/content/drive/MyDrive/chroma_db`

Secrets required in Colab:
- `GEMINI_API_KEY`
- `NGROK_AUTH_TOKEN`


In [ ]:
# Install all dependencies used by the full project
!pip install -q pandas numpy tqdm chromadb langchain-text-splitters google-genai streamlit python-dotenv pyngrok

print("✅ Dependencies installed.")


In [ ]:
# Mount Google Drive and load secrets

from google.colab import drive, userdata
from pathlib import Path
import os

drive.mount('/content/drive')

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')

if not GEMINI_API_KEY:
    raise ValueError(
        'GEMINI_API_KEY is missing. Add it in Colab Secrets.'
    )

if not NGROK_AUTH_TOKEN:
    raise ValueError(
        'NGROK_AUTH_TOKEN is missing. Add it in Colab Secrets.'
    )

os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
os.environ['CHROMA_DB_PATH'] = '/content/drive/MyDrive/chroma_db'

DATASET_PATH = Path('/content/drive/MyDrive/postings.csv')
CHROMA_DB_PATH = Path('/content/drive/MyDrive/chroma_db')
COLLECTION_NAME = 'tech_jobs'

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f'Dataset not found at {DATASET_PATH}. '
        'Make sure postings.csv is already in My Drive.'
    )

CHROMA_DB_PATH.mkdir(parents=True, exist_ok=True)

print("✅ Google Drive mounted.")
print("✅ Gemini API key loaded from Colab Secrets.")
print("✅ ngrok token loaded from Colab Secrets.")
print(f"✅ Dataset: {DATASET_PATH}")
print(f"✅ ChromaDB: {CHROMA_DB_PATH}")


## Stage 1 — Build / Reuse the ChromaDB Job Index

The index is reused when it already contains data. Set `REBUILD_INDEX = True` only when you intentionally want to rebuild it.


In [ ]:
# Stage 1: load, clean, chunk, and index the job dataset

import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from chromadb.utils import embedding_functions
import uuid

MAX_ROWS = 10_000
BATCH_SIZE = 100
REBUILD_INDEX = False

df = pd.read_csv(DATASET_PATH).head(MAX_ROWS)

print(f"Loaded {len(df):,} job postings.")
print(f"Columns: {list(df.columns)}")

def clean_text(text) -> str:
    if pd.isna(text):
        return ''
    return str(text).replace('\n', ' ').strip()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=40,
    separators=['\n\n', '\n', '. ', ' ', '']
)

embedding_fn = embedding_functions.DefaultEmbeddingFunction()
client = chromadb.PersistentClient(path=str(CHROMA_DB_PATH))

try:
    collection = client.get_collection(
        name=COLLECTION_NAME,
        embedding_function=embedding_fn
    )
    print(
        f"Existing collection found: {COLLECTION_NAME} "
        f"({collection.count():,} chunks)"
    )
except Exception:
    collection = client.get_or_create_collection(
        name=COLLECTION_NAME,
        embedding_function=embedding_fn,
        metadata={'hnsw:space': 'cosine'}
    )
    print(f"Created collection: {COLLECTION_NAME}")

if REBUILD_INDEX:
    try:
        client.delete_collection(name=COLLECTION_NAME)
    except Exception:
        pass

    collection = client.get_or_create_collection(
        name=COLLECTION_NAME,
        embedding_function=embedding_fn,
        metadata={'hnsw:space': 'cosine'}
    )

    print("Existing collection deleted. Fresh index will be built.")

def index_jobs(dataframe: pd.DataFrame, batch_size: int = 100) -> int:
    documents_batch = []
    metadatas_batch = []
    ids_batch = []
    total_chunks = 0

    for idx, row in dataframe.iterrows():

        job_title = row.get('title', 'Unknown Title')
        company = row.get('company', 'Unknown Company')
        description = clean_text(row.get('description', ''))

        if not description:
            continue

        parent_id = f'job_{uuid.uuid4().hex[:8]}'
        chunks = text_splitter.split_text(description)

        for chunk_index, chunk in enumerate(chunks):

            documents_batch.append(chunk)

            metadatas_batch.append({
                'parent_id': parent_id,
                'job_title': (
                    str(job_title)
                    if not pd.isna(job_title)
                    else 'Unknown Title'
                ),
                'company': (
                    str(company)
                    if not pd.isna(company)
                    else 'Unknown Company'
                ),
                'chunk_index': chunk_index,
                'total_chunks': len(chunks)
            })

            ids_batch.append(
                f'{parent_id}_chunk_{chunk_index}'
            )

            if len(documents_batch) >= batch_size:

                collection.add(
                    documents=documents_batch,
                    metadatas=metadatas_batch,
                    ids=ids_batch
                )

                total_chunks += len(documents_batch)
                documents_batch.clear()
                metadatas_batch.clear()
                ids_batch.clear()

        if (idx + 1) % 500 == 0:
            print(
                f"Processed {idx + 1:,}/{len(dataframe):,} rows..."
            )

    if documents_batch:

        collection.add(
            documents=documents_batch,
            metadatas=metadatas_batch,
            ids=ids_batch
        )

        total_chunks += len(documents_batch)

    return total_chunks

if REBUILD_INDEX or collection.count() == 0:

    total_chunks = index_jobs(
        df,
        batch_size=BATCH_SIZE
    )

    print(f"\nDone. Added {total_chunks:,} chunks.")

else:

    print(
        "\nSkipped indexing because a ChromaDB index already exists."
    )

print(f"Collection size: {collection.count():,}")
print(f"ChromaDB path: {CHROMA_DB_PATH}")


## Stage 2 — Semantic Search + RAG Career Advisor


In [ ]:
# Stage 2: semantic retrieval + grounded Gemini response

from google import genai
from google.genai import types

TOP_K = 3
MAX_DISTANCE = 0.8

CANDIDATE_MODELS = [
    'models/gemini-3.5-flash',
    'models/gemini-2.5-flash',
    'models/gemini-2.5-pro',
    'models/gemini-flash-latest',
]

class ChromaJobRetriever:
    """Semantic retrieval over the persistent ChromaDB collection."""

    def __init__(self, db_path: Path, collection_name: str):

        if not db_path.exists():
            raise FileNotFoundError(
                f'ChromaDB not found: {db_path}. Run Stage 1 first.'
            )

        self.client = chromadb.PersistentClient(
            path=str(db_path)
        )

        self.embedding_fn = (
            embedding_functions.DefaultEmbeddingFunction()
        )

        self.collection = self.client.get_collection(
            name=collection_name,
            embedding_function=self.embedding_fn
        )

    def search(
        self,
        query: str,
        top_k: int = TOP_K,
        company_filter: str | None = None,
    ):

        if not query or not query.strip():
            raise ValueError('Query must not be empty.')

        where_clause = (
            {'company': company_filter}
            if company_filter
            else None
        )

        return self.collection.query(
            query_texts=[query.strip()],
            n_results=top_k,
            where=where_clause
        )


class RAGJobGenerator:
    """Generate a grounded career-advisor response from retrieved job context."""

    def __init__(
        self,
        api_key: str,
        candidate_models: list[str] | None = None
    ):

        if not api_key:
            raise ValueError(
                'GEMINI_API_KEY is missing.'
            )

        self.client = genai.Client(
            api_key=api_key
        )

        self.candidate_models = (
            candidate_models
            or CANDIDATE_MODELS
        )

    def build_prompt(
        self,
        query: str,
        retrieved_results
    ) -> str:

        documents = retrieved_results['documents'][0]
        metadatas = retrieved_results['metadatas'][0]
        distances = retrieved_results['distances'][0]

        context_blocks = []

        for doc, meta, dist in zip(
            documents,
            metadatas,
            distances
        ):

            if dist > MAX_DISTANCE:
                continue

            context_blocks.append(
                f"- Job title: {meta.get('job_title')}\n"
                f"  Company: {meta.get('company')}\n"
                f"  Details: {doc}"
            )

        context_text = (
            '\n\n'.join(context_blocks)
            if context_blocks
            else (
                'No sufficiently relevant jobs were found '
                'in the current database.'
            )
        )

        return f"""
You are an expert AI Career Advisor analyzing tech job market data.

Answer the user's question STRICTLY based on the provided Job Context below.

Do not invent jobs, companies, requirements, locations, or other facts.

If the context does not contain relevant information, say:
"I couldn't find relevant jobs matching your criteria in the current database."

### Job Context
{context_text}

### User Query
{query}

### Answer
"""

    def generate_answer(self, prompt: str) -> str:

        config = types.GenerateContentConfig(
            temperature=0.1
        )

        last_error = None

        for model in self.candidate_models:

            try:

                print(f"Trying model: {model}")

                response = (
                    self.client.models.generate_content(
                        model=model,
                        contents=prompt,
                        config=config
                    )
                )

                if response.text:
                    return response.text

            except Exception as exc:

                last_error = exc
                print(
                    f"Model unavailable: {model}"
                )

        raise RuntimeError(
            'All configured Gemini models failed.'
        ) from last_error


retriever = ChromaJobRetriever(
    db_path=CHROMA_DB_PATH,
    collection_name=COLLECTION_NAME
)

print("✅ ChromaDB collection loaded.")
print(
    f"Number of indexed chunks: "
    f"{retriever.collection.count():,}"
)

USER_QUERY = "Python developer with machine learning experience"

search_results = retriever.search(
    USER_QUERY,
    top_k=TOP_K
)

documents = search_results['documents'][0]
metadatas = search_results['metadatas'][0]
distances = search_results['distances'][0]

print(f"\nFound {len(documents)} candidate chunks")
print("=" * 60)

for idx, (doc, meta, dist) in enumerate(
    zip(documents, metadatas, distances),
    start=1
):

    print(
        f"\nResult #{idx} | Cosine distance: {dist:.4f}"
    )

    print(
        f"Company: {meta.get('company')}"
    )

    print(
        f"Job title: {meta.get('job_title')}"
    )

    print(
        f"Parent ID: {meta.get('parent_id')}"
    )

    print(
        f"Text: {doc}"
    )

    print("-" * 60)


generator = RAGJobGenerator(
    api_key=GEMINI_API_KEY
)

prompt = generator.build_prompt(
    USER_QUERY,
    search_results
)

ai_response = generator.generate_answer(prompt)

print("\n" + "=" * 60)
print("Final AI Response")
print("=" * 60)
print(ai_response)


## Streamlit Interface

The final step downloads the GitHub version of `app.py`, launches Streamlit, and exposes it through ngrok.

No API key is written into `app.py`; the key is supplied through Colab Secrets.


In [ ]:
# Download the GitHub app.py and run Streamlit

import subprocess
import time
from pyngrok import ngrok

REPO_URL = "https://github.com/rashedkarnoub661-cyber/tech-jobs-analyzer.git"
REPO_DIR = "/content/tech-jobs-analyzer"

# Clone the repository if it is not already present
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        check=True
    )

APP_PATH = os.path.join(REPO_DIR, "app.py")

if not os.path.exists(APP_PATH):
    raise FileNotFoundError(
        f"app.py was not found in the GitHub repository: {APP_PATH}"
    )

# Environment variables used by app.py
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ["CHROMA_DB_PATH"] = str(CHROMA_DB_PATH)

# Start Streamlit
streamlit_process = subprocess.Popen(
    [
        "streamlit",
        "run",
        APP_PATH,
        "--server.port=8501",
        "--server.address=0.0.0.0",
        "--server.headless=true",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)

print("✅ Streamlit started on port 8501.")

# Configure ngrok and create the public URL
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Close an existing tunnel on this runtime if present
try:
    ngrok.kill()
except Exception:
    pass

public_url = ngrok.connect(8501)

print("✅ Full project is running.")
print("🌐 Streamlit URL:")
print(public_url)
